# Data Mining Assignment — Apache Spark Data Lake Preprocessing

**Name:** Sidharth Kumar Singh &nbsp;&nbsp;|&nbsp;&nbsp; **Roll Number:** LCS2023043

**Environment:** Google Colab + PySpark

This notebook builds a minimal data lake for `sales.csv`:

- **Bronze** keeps the raw dataset exactly as provided (no changes).
- **Silver** contains the preprocessed dataset, produced with Apache Spark.

Run the notebook from top to bottom (`Runtime → Run all`).


In [ ]:
# Install PySpark in the Colab runtime (nothing is installed outside Colab).
!pip -q install pyspark


In [ ]:
import hashlib
import os
import shutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

spark = (SparkSession.builder
         .appName("DataLakePreprocessing")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


## 1. Data Lake structure (Bronze layer)

The raw dataset is placed in the Bronze layer and is never modified:

```
data_lake/
|-- bronze/
|   `-- sales.csv
`-- silver/
    `-- preprocessed_sales/
```


In [ ]:
DATA_LAKE = "data_lake"
BRONZE = os.path.join(DATA_LAKE, "bronze")
SILVER = os.path.join(DATA_LAKE, "silver")
os.makedirs(BRONZE, exist_ok=True)
os.makedirs(SILVER, exist_ok=True)

bronze_path = os.path.join(BRONZE, "sales.csv")
RAW_URL = ("https://raw.githubusercontent.com/necromancersanju-netizen/"
           "data-mining-data-lake-assignment/main/dataset/sales.csv")

if not os.path.exists(bronze_path):
    try:
        import urllib.request
        urllib.request.urlretrieve(RAW_URL, bronze_path)
    except Exception as err:
        print("Automatic download failed:", err)

if not os.path.exists(bronze_path):  # fallback: upload the provided dataset manually
    from google.colab import files
    uploaded = files.upload()
    shutil.move(next(iter(uploaded)), bronze_path)

with open(bronze_path, "rb") as fh:
    bronze_md5 = hashlib.md5(fh.read()).hexdigest()
with open(bronze_path, newline="") as fh:
    bronze_rows = sum(1 for _ in fh) - 1

print("Bronze file      :", bronze_path, "(raw, unchanged)")
print("Bronze MD5       :", bronze_md5)
print("Bronze data rows :", bronze_rows)


## 2. Load and inspect the Bronze data with Spark

Every column is read as text first, so that format problems are detected with
Spark operations instead of failing at read time.


In [ ]:
df = (spark.read
      .option("header", True)
      .option("inferSchema", False)
      .csv(bronze_path))

df.printSchema()
df.show(5, truncate=False)

raw_count = df.count()
print("Total records in Bronze:", raw_count)


### 2.1 Duplicate rows

Some `order_id` values appear twice with different order details — those are
real, distinct records. Only *exact* duplicate rows are counted here and
removed later.


In [ ]:
total = df.count()
distinct_rows = df.distinct().count()
dup_groups = df.groupBy(*df.columns).count().filter(F.col("count") > 1).count()

print("Exact duplicate row groups :", dup_groups)
print("Extra duplicate rows       :", total - distinct_rows)
df.groupBy(*df.columns).count().filter(F.col("count") > 1).orderBy("order_id").show(truncate=False)


### 2.2 Missing values

In [ ]:
def blank_or_null(column):
    return F.col(column).isNull() | (F.trim(F.col(column)) == "")

print("Rows with missing/empty values per column:")
df.select([F.sum(F.when(blank_or_null(c), 1).otherwise(0)).alias(c)
           for c in df.columns]).show(truncate=False)


### 2.3 Text-format inconsistencies (whitespace and capitalization)

In [ ]:
text_cols = [c for c, t in df.dtypes if t == "string"]

print("Rows with extra whitespace per column:")
df.select([F.sum(F.when(F.col(c) != F.trim(F.col(c)), 1).otherwise(0)).alias(c)
           for c in text_cols]).show(truncate=False)

for c in ("category", "city"):
    print(f"Values of '{c}' that differ only by capitalization:")
    (df.filter(F.col(c).isNotNull())
       .groupBy(c).count()
       .withColumn("normalized", F.lower(F.col(c)))
       .groupBy("normalized")
       .agg(F.collect_set(c).alias("variants"), F.sum("count").alias("rows"))
       .filter(F.size("variants") > 1)
       .show(truncate=False))


### 2.4 Invalid numerical values

In [ ]:
NUMBER_PATTERN = r"^-?\d+(\.\d+)?$"
numeric_cols = ["quantity", "unit_price", "discount_percent"]

for c in numeric_cols:
    raw = F.trim(F.col(c))
    parsed = F.when(raw.rlike(NUMBER_PATTERN), raw.cast("double"))
    bad_format = (~raw.rlike(NUMBER_PATTERN)) | F.col(c).isNull()
    if c == "discount_percent":
        out_of_range = (~bad_format) & ((parsed < 0) | (parsed > 100))
    else:
        out_of_range = (~bad_format) & (parsed <= 0)

    row = df.select(
        F.sum(F.when(bad_format, 1).otherwise(0)).alias("non_numeric"),
        F.sum(F.when(out_of_range, 1).otherwise(0)).alias("out_of_range"),
        F.min(parsed).alias("min"),
        F.max(parsed).alias("max"),
    ).first()
    print(f"{c}: format_errors={row['non_numeric']} invalid_values={row['out_of_range']} "
          f"min={row['min']} max={row['max']}")


### 2.5 Inconsistent / invalid dates

In [ ]:
DATE_FORMATS = ["yyyy-MM-dd", "d/M/yyyy", "yyyy/M/d", "d-M-yyyy", "M-d-yyyy"]


def parse_date(column):
    raw = F.trim(F.col(column))
    return F.coalesce(*[F.try_to_date(raw, fmt) for fmt in DATE_FORMATS])


dates = df.select("order_date", parse_date("order_date").alias("parsed"),
                  F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$").alias("iso_format"))

print("Rows in ISO format          :", dates.filter(F.col("iso_format")).count())
print("Rows in other valid formats :",
      dates.filter(~F.col("iso_format") & F.col("parsed").isNotNull()).count())
print("Rows with invalid dates     :", dates.filter(F.col("parsed").isNull()).count())
print("Non-ISO values:")
dates.filter(~F.col("iso_format")).show(truncate=False)


## 3. Basic preprocessing — Silver layer

In [ ]:
# Step 1: remove exact duplicate rows.
silver = df.dropDuplicates()
print("Rows after removing exact duplicates:", silver.count())


In [ ]:
# Step 2: handle missing values with a simple marker.
FILL_VALUE = "Unknown"
for c in ["customer_name", "payment_method", "city"]:
    silver = silver.withColumn(
        c, F.when(blank_or_null(c), F.lit(FILL_VALUE)).otherwise(F.col(c)))
print("Missing values filled with:", FILL_VALUE)


In [ ]:
# Step 3: trim extra spaces in all text fields and standardize capitalization
# in the fields where inconsistencies were found (category, city).
for c in [c for c, t in silver.dtypes if t == "string"]:
    silver = silver.withColumn(c, F.trim(F.col(c)))
for c in ["category", "city"]:
    silver = silver.withColumn(c, F.initcap(F.col(c)))


In [ ]:
# Step 4: validate quantity / unit_price / discount_percent.
# Clearly invalid values are set to null, then imputed with the column median
# (the median is computed from the data with Spark, not hard-coded).
for c in numeric_cols:
    raw = F.trim(F.col(c))
    parsed = F.when(raw.rlike(NUMBER_PATTERN), raw.cast("double"))
    if c == "discount_percent":
        valid = raw.rlike(NUMBER_PATTERN) & (parsed >= 0) & (parsed <= 100)
    else:
        valid = raw.rlike(NUMBER_PATTERN) & (parsed > 0)
    silver = silver.withColumn(c, F.when(valid, parsed).otherwise(F.lit(None).cast("double")))

medians = {c: silver.select(F.percentile_approx(F.col(c), 0.5)).first()[0] for c in numeric_cols}
print("Medians used for imputation:", medians)

silver = (silver
          .withColumn("quantity",
                      F.when(F.col("quantity").isNull(), F.lit(medians["quantity"]))
                       .otherwise(F.col("quantity")).cast(IntegerType()))
          .withColumn("discount_percent",
                      F.when(F.col("discount_percent").isNull(), F.lit(medians["discount_percent"]))
                       .otherwise(F.col("discount_percent")).cast(IntegerType()))
          .withColumn("unit_price",
                      F.when(F.col("unit_price").isNull(), F.lit(medians["unit_price"]))
                       .otherwise(F.col("unit_price")).cast(DoubleType())))

silver.agg(F.min("quantity").alias("min_quantity"),
           F.min("unit_price").alias("min_unit_price"),
           F.min("discount_percent").alias("min_discount")).show()


In [ ]:
# Step 5: normalize valid dates to ISO yyyy-MM-dd. Values that are not valid
# calendar dates are set to null (left empty) instead of inventing a date;
# the rest of the record is still useful, so the row is kept.
silver = silver.withColumn("parsed_date", parse_date("order_date"))
bad_dates = silver.filter(F.col("parsed_date").isNull()).count()
print("Rows with a clearly invalid date (left empty):", bad_dates)
silver.filter(F.col("parsed_date").isNull()).show(truncate=False)

silver = (silver
          .drop("order_date")
          .withColumnRenamed("parsed_date", "order_date"))

# Keep the original column order.
silver = silver.select(*df.columns)
silver.show(5, truncate=False)


## 4. Save the Silver layer

In [ ]:
silver_output = os.path.join(SILVER, "preprocessed_sales")
(silver.coalesce(1)
       .write.mode("overwrite")
       .option("header", True)
       .csv(silver_output))

part_file = [f for f in os.listdir(silver_output)
             if f.startswith("part-") and f.endswith(".csv")][0]
silver_csv = os.path.join(SILVER, "sales_silver.csv")
shutil.copy(os.path.join(silver_output, part_file), silver_csv)

print("Spark output folder :", silver_output)
print("Single CSV copy     :", silver_csv)


## 5. Verification

In [ ]:
final_count = silver.count()
print("Rows before preprocessing (Bronze):", raw_count)
print("Rows after preprocessing (Silver) :", final_count)
print("Rows removed in total             :", raw_count - final_count)

print("\nSilver data read back with Spark:")
silver_check = (spark.read
                .option("header", True)
                .option("inferSchema", True)
                .csv(silver_csv))
silver_check.printSchema()
silver_check.show(5, truncate=False)

remaining_duplicates = silver_check.count() - silver_check.distinct().count()
remaining_nulls = silver_check.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
     for c in silver_check.columns]).first()
print("Remaining duplicate rows:", remaining_duplicates)
print("Remaining null values   :", remaining_nulls.asDict())


In [ ]:
print("""
Preprocessing performed (Bronze -> Silver):
1. Removed exact duplicate rows.
2. Filled missing customer_name / payment_method / city values with 'Unknown'.
3. Trimmed extra spaces and normalized capitalization in the affected text
   fields (category, city).
4. Validated quantity (> 0), unit_price (> 0) and discount_percent (0-100):
   clearly invalid values were set to null and imputed with the column median.
5. Converted valid dates from several formats to ISO yyyy-MM-dd; values that
   are not valid calendar dates were left empty (the records are preserved).
6. Saved the resulting DataFrame as CSV in the Silver layer
   (data_lake/silver/preprocessed_sales, with a single-file copy
   sales_silver.csv for submission).
""")
